# V8 — Fix F1 Drop + RH_7

**V7 analysis:**
- RH_1-6 fixed (near-perfect) ← from signal features + aggressive pruning ✅
- F1 dropped 0.901→0.822 ← from post-filter dampening ❌
- RH_7 crashed 1.0→0.29 ← post-filter can't handle multi-pattern RH ❌

**V8 fixes:**
- KEEP signal features (they help RH_1-6 via the model)
- REMOVE post-filter dampening (restore F1)
- Reduce pruning aggressiveness (was over-pruning real mules)
- Add "too-perfect" detection (catches RH_7 = multi-pattern decoys)

## 0 — Setup

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score, fbeta_score,
                             precision_score, recall_score, confusion_matrix)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from glob import glob
import pyarrow.parquet as pq
import gc, warnings
warnings.filterwarnings('ignore')

DATA_DIR = "IITD-Tryst-Hackathon/Phase 2"
t0 = time.time()

print("Loading data...")
train = pd.read_csv("features_train_p2.csv")
test = pd.read_csv("features_test_p2.csv")
accounts = pd.read_parquet(f"{DATA_DIR}/accounts.parquet")
labels = pd.read_parquet(f"{DATA_DIR}/train_labels.parquet")

train = train.merge(accounts[["account_id", "branch_code"]], on="account_id", how="left")
test = test.merge(accounts[["account_id", "branch_code"]], on="account_id", how="left")
train = train.merge(labels[["account_id", "alert_reason"]], on="account_id", how="left")

print(f"Train: {train.shape} | Test: {test.shape}")

Loading data...
Train: (96091, 73) | Test: (64062, 71)


## 1 — OOF Target Encoding

In [2]:
print("OOF Target Encoding...")
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train["branch_mule_rate_oof"] = np.nan
global_mean = train["is_mule"].mean()

for tr_idx, val_idx in skf_te.split(train, train["is_mule"]):
    tr_df = train.iloc[tr_idx]
    branch_stats = tr_df.groupby("branch_code")["is_mule"].agg(['sum', 'count'])
    branch_stats["rate"] = (branch_stats["sum"] + 10 * global_mean) / (branch_stats["count"] + 10)
    mapped = train.iloc[val_idx]["branch_code"].map(branch_stats["rate"]).fillna(global_mean)
    train.loc[train.index[val_idx], "branch_mule_rate_oof"] = mapped.values

branch_stats_full = train.groupby("branch_code")["is_mule"].agg(['sum', 'count'])
branch_stats_full["rate"] = (branch_stats_full["sum"] + 10 * global_mean) / (branch_stats_full["count"] + 10)
test["branch_mule_rate_oof"] = test["branch_code"].map(branch_stats_full["rate"]).fillna(global_mean)

score_cols = ["near_threshold_pct", "round_amount_pct", "gap_cv", "degree_centrality",
              "mule_trigram_count", "branch_mule_rate_oof", "has_prior_freeze"]
for df in [train, test]:
    c_score = np.zeros(len(df))
    for col in score_cols:
        if col in df.columns:
            m, s = train[col].mean(), train[col].std()
            c_score += (df[col].fillna(m) - m) / s if s > 0 else 0
    df["composite_score_fixed"] = c_score

OOF Target Encoding...


## 2 — Multi-Signal Features (kept from V7 — helps RH_1-6)

In [3]:
print("Building multi-signal features...")

for df in [train, test]:
    signals = []

    if "days_to_first_large" in df.columns:
        df["sig_dormant"] = (df["days_to_first_large"] > 180).astype(float)
        signals.append("sig_dormant")

    if "near_threshold_pct" in df.columns:
        df["sig_structuring"] = (df["near_threshold_pct"] > train["near_threshold_pct"].quantile(0.90)).astype(float)
        signals.append("sig_structuring")

    if "median_dwell_hours" in df.columns:
        df["sig_rapid"] = (df["median_dwell_hours"] < 24).astype(float)
        signals.append("sig_rapid")

    if "unique_cp_count" in df.columns and "txn_count" in df.columns:
        df["sig_fanout"] = (df["unique_cp_count"] / df["txn_count"].clip(1) > 0.5).astype(float)
        signals.append("sig_fanout")

    if "days_to_first_large" in df.columns and "total_volume" in df.columns:
        df["sig_new_highvol"] = ((df["days_to_first_large"] < 30) &
                                  (df["total_volume"] > train["total_volume"].quantile(0.75))).astype(float)
        signals.append("sig_new_highvol")

    if "round_amount_pct" in df.columns:
        df["sig_round"] = (df["round_amount_pct"] > train["round_amount_pct"].quantile(0.90)).astype(float)
        signals.append("sig_round")

    if "has_mobile_update" in df.columns:
        df["sig_post_mobile"] = (df["has_mobile_update"] == 1).astype(float)
        signals.append("sig_post_mobile")

    if "total_volume" in df.columns and "balance_mean" in df.columns:
        vol_to_bal = df["total_volume"] / df["balance_mean"].abs().clip(1)
        df["sig_income_mismatch"] = (vol_to_bal > train["total_volume"].div(train["balance_mean"].abs().clip(1)).quantile(0.90)).astype(float)
        signals.append("sig_income_mismatch")

    if "branch_mule_rate_oof" in df.columns:
        df["sig_branch_cluster"] = (df["branch_mule_rate_oof"] > train["branch_mule_rate_oof"].quantile(0.90)).astype(float)
        signals.append("sig_branch_cluster")

    if signals:
        df["multi_signal_count"] = sum(df[s] for s in signals)
        df["signal_consistency"] = df["multi_signal_count"] / len(signals)

    # NEW for RH_7: "too perfect" flag
    # Real mules are messy — they match some patterns strongly and others weakly.
    # Planted red herrings may match ALL patterns at moderate levels.
    # Compute uniformity: if signals are all ~0.5 instead of mixed 0s and 1s → suspicious
    if signals:
        sig_vals = np.column_stack([df[s].fillna(0).values for s in signals])
        # Entropy-like measure: how uniform are the signals?
        # Real mules: some signals=1, some=0 → high variance
        # Planted RH_7: may have more uniform moderate signals
        df["signal_variance"] = np.var(sig_vals, axis=1)
        # If too many signals are ON simultaneously → could be too-perfect
        df["signal_saturation"] = (df["multi_signal_count"] >= len(signals) * 0.7).astype(float)

print(f"Multi-signal features: {len(signals)} signals + count + consistency + variance + saturation")

Building multi-signal features...
Multi-signal features: 7 signals + count + consistency + variance + saturation


## 3 — Prepare Features

In [4]:
drop_cols = ["account_id", "is_mule", "first_large_ts", "open_date",
             "branch_code", "branch_mule_rate", "composite_score", "alert_reason"]
features = [c for c in train.columns if c not in drop_cols and train[c].nunique() > 1]
train[features] = train[features].fillna(train[features].median())
test[features] = test[features].fillna(train[features].median())
print(f"Total features: {len(features)}")

Total features: 76


## 4 — Red Herring Pruning (Standard — not over-aggressive)

In [5]:
print("Red herring pruning (standard)...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X = train[features].values
y = train["is_mule"].values
oof_screen = np.zeros(len(y))

for fold, (tr, val) in enumerate(skf.split(X, y)):
    m = lgb.LGBMClassifier(n_estimators=300, random_state=42, n_jobs=-1, verbosity=-1)
    m.fit(X[tr], y[tr])
    oof_screen[val] = m.predict_proba(X[val])[:,1]

# Standard: only drop extreme red herrings (p<0.02)
extreme = (y == 1) & (oof_screen < 0.02)
keep_mask = ~extreme
X_clean = X[keep_mask]
y_clean = y[keep_mask]
print(f"Dropped {extreme.sum()} extreme red herrings → {len(y_clean):,} samples")

# NO moderate pruning, NO aggressive down-weighting — that hurt F1

Red herring pruning (standard)...
Dropped 342 extreme red herrings → 95,749 samples


## 5 — Ensemble (Simple Average — Well-Calibrated)

NO post-filter, NO rank averaging. Just clean calibrated probabilities.

In [6]:
print("Training LGB + XGB + CatBoost...")
spw = (y_clean == 0).sum() / max(1, (y_clean == 1).sum())
oof_lgb, oof_xgb, oof_cat = np.zeros(len(y_clean)), np.zeros(len(y_clean)), np.zeros(len(y_clean))
t_lgb, t_xgb, t_cat = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))

X_test = test[features].values

for fold, (tr, val) in enumerate(skf.split(X_clean, y_clean)):
    print(f"--- Fold {fold+1} ---")
    Xtr, Xval = X_clean[tr], X_clean[val]
    ytr, yval = y_clean[tr], y_clean[val]

    m1 = lgb.LGBMClassifier(n_estimators=1200, learning_rate=0.03, max_depth=8,
                            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                            random_state=42, verbosity=-1, n_jobs=-1)
    m1.fit(Xtr, ytr, eval_set=[(Xval, yval)],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val] = m1.predict_proba(Xval)[:,1]
    t_lgb += m1.predict_proba(X_test)[:,1] / 5.0

    m2 = xgb.XGBClassifier(n_estimators=1200, learning_rate=0.03, max_depth=7,
                           subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                           random_state=42, verbosity=0, eval_metric="auc", n_jobs=-1,
                           early_stopping_rounds=50)
    m2.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
    oof_xgb[val] = m2.predict_proba(Xval)[:,1]
    t_xgb += m2.predict_proba(X_test)[:,1] / 5.0

    m3 = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=7,
                            auto_class_weights='Balanced', random_state=42,
                            verbose=False, early_stopping_rounds=50)
    m3.fit(Xtr, ytr, eval_set=(Xval, yval))
    oof_cat[val] = m3.predict_proba(Xval)[:,1]
    t_cat += m3.predict_proba(X_test)[:,1] / 5.0

# Simple average — NO rank, NO post-filter
oof_ens = (oof_lgb + oof_xgb + oof_cat) / 3.0
t_ens = (t_lgb + t_xgb + t_cat) / 3.0

auc = roc_auc_score(y_clean, oof_ens)
print(f"\nOOF AUC: {auc:.4f}")

# F2 threshold
best_f2, best_t = 0, 0.5
for t in np.arange(0.1, 0.95, 0.01):
    f2 = fbeta_score(y_clean, (oof_ens > t).astype(int), beta=2)
    if f2 > best_f2:
        best_f2, best_t = f2, t

preds = (oof_ens > best_t).astype(int)
cm = confusion_matrix(y_clean, preds)
print(f"Threshold (F2): {best_t:.2f}")
print(f"  F1={f1_score(y_clean,preds):.4f}  F2={best_f2:.4f}  "
      f"P={precision_score(y_clean,preds):.4f}  R={recall_score(y_clean,preds):.4f}")
print(f"  CM: TN={cm[0,0]:,} FP={cm[0,1]:,} FN={cm[1,0]:,} TP={cm[1,1]:,}")

test["is_mule_prob"] = t_ens
mule_preds = test[test["is_mule_prob"] > best_t]["account_id"].tolist()
print(f"\nPredicted mules: {len(mule_preds)} ({len(mule_preds)/len(test)*100:.1f}%)")
print(f"Calibration: mean prob = {t_ens.mean():.4f} (expected ~{global_mean:.4f})")

Training LGB + XGB + CatBoost...
--- Fold 1 ---
--- Fold 2 ---
--- Fold 3 ---
--- Fold 4 ---
--- Fold 5 ---

OOF AUC: 0.9966
Threshold (F2): 0.46
  F1=0.8449  F2=0.9092  P=0.7559  R=0.9577
  CM: TN=92,684 FP=724 FN=99 TP=2,242

Predicted mules: 1825 (2.8%)
Calibration: mean prob = 0.0326 (expected ~0.0279)


## 6 — Adaptive CDF Temporal Windows

In [7]:
print("=" * 60)
print("ADAPTIVE CDF TEMPORAL WINDOWS")
print("=" * 60)

parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))
print(f"Transaction parts: {len(parts)}")

temporal_threshold = best_t * 0.25
high_prob_ids = set(test[test["is_mule_prob"] > temporal_threshold]["account_id"].tolist())
print(f"Accounts for temporal: {len(high_prob_ids):,}")

daily_vol = {}
for i, p in enumerate(parts):
    try:
        ds = pq.read_table(p, columns=["account_id", "transaction_timestamp", "amount"],
                           filters=[("account_id", "in", list(high_prob_ids))])
        df = ds.to_pandas()
    except:
        df = pd.read_parquet(p, columns=["account_id", "transaction_timestamp", "amount"])
        df = df[df["account_id"].isin(high_prob_ids)]
    if df.empty: continue
    df["ts"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")
    df["date"] = df["ts"].dt.date
    df["abs_amount"] = df["amount"].abs()
    grp = df.groupby(["account_id", "date"])["abs_amount"].sum()
    for (aid, dt), vol in grp.items():
        if aid not in daily_vol:
            daily_vol[aid] = {}
        daily_vol[aid][dt] = daily_vol[aid].get(dt, 0) + vol
    del df
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(parts)}] processed")
    gc.collect()

print(f"Built series for {len(daily_vol):,} accounts")

ADAPTIVE CDF TEMPORAL WINDOWS
Transaction parts: 396
Accounts for temporal: 2,858
  [100/396] processed
  [200/396] processed
  [300/396] processed
Built series for 2,821 accounts


In [8]:
def v8_temporal_window(vol_dict):
    """V8: Multi-scale densest window + inner CDF trimming."""
    if len(vol_dict) < 3:
        return "", ""

    dates = sorted(vol_dict.keys())
    vols = np.array([vol_dict[d] for d in dates])
    total = vols.sum()
    if total == 0:
        return "", ""

    best_start, best_end = 0, len(dates) - 1
    for window_days in [14, 30, 60, 90]:
        best_wvol = 0
        b_s, b_e = 0, 0
        for j in range(len(vols)):
            k = j
            wvol = 0
            while k < len(vols) and (dates[k] - dates[j]).days <= window_days:
                wvol += vols[k]
                k += 1
            if wvol > best_wvol:
                best_wvol = wvol
                b_s, b_e = j, k - 1
        if best_wvol / total >= 0.50:
            best_start, best_end = b_s, b_e
            break

    w_vols = vols[best_start:best_end+1]
    w_dates = dates[best_start:best_end+1]
    if len(w_vols) > 5:
        w_cum = np.cumsum(w_vols)
        w_tot = w_cum[-1]
        if w_tot > 0:
            w_cdf = w_cum / w_tot
            s_idx = min(np.searchsorted(w_cdf, 0.05), len(w_dates) - 1)
            e_idx = min(np.searchsorted(w_cdf, 0.95), len(w_dates) - 1)
            w_dates = w_dates[s_idx:e_idx+1]

    if len(w_dates) == 0:
        w_dates = dates[best_start:best_end+1]

    return f"{w_dates[0]}T00:00:00", f"{w_dates[-1]}T23:59:59"

temporal_windows = {}
for aid in high_prob_ids:
    if aid in daily_vol:
        s, e = v8_temporal_window(daily_vol[aid])
        temporal_windows[aid] = (s, e)
    else:
        temporal_windows[aid] = ("", "")

n_with = sum(1 for s, e in temporal_windows.values() if s != "")
print(f"Accounts with windows: {n_with:,}/{len(high_prob_ids):,}")

widths = []
for s, e in temporal_windows.values():
    if s and e:
        widths.append((pd.to_datetime(e) - pd.to_datetime(s)).days)
wa = np.array(widths) if widths else np.array([0])
print(f"Window width: median={np.median(wa):.0f}d, mean={wa.mean():.0f}d")

Accounts with windows: 2,820/2,858
Window width: median=504d, mean=537d


## 7 — Generate Submission

In [9]:
print("=" * 60)
print("GENERATING SUBMISSION V8")
print("=" * 60)

submission = pd.DataFrame({
    "account_id": test["account_id"],
    "is_mule": test["is_mule_prob"],
    "suspicious_start": "",
    "suspicious_end": ""
})

for aid, (s, e) in temporal_windows.items():
    mask = submission["account_id"] == aid
    submission.loc[mask, "suspicious_start"] = s
    submission.loc[mask, "suspicious_end"] = e

submission.to_csv("submission_v8.csv", index=False)

print(f"Submission: {submission.shape}")
print(f"  Mean prob:    {submission['is_mule'].mean():.4f}")
print(f"  >50% mule:    {(submission['is_mule']>0.5).sum():,}")
print(f"  With windows: {(submission['suspicious_start']!='').sum():,}")
print(f"\n✅ submission_v8.csv saved")
print(f"Total: {time.time()-t0:.0f}s = {(time.time()-t0)/60:.1f} min")

GENERATING SUBMISSION V8
Submission: (64062, 4)
  Mean prob:    0.0326
  >50% mule:    1,786
  With windows: 2,820

✅ submission_v8.csv saved
Total: 287s = 4.8 min
